# Task 6 – Wu-Palmer + phonetic similarity

> *We would like to extend the Wu and Palmer semantic similarity extension of Lesk algorithm implemented in 3)
> by incorporating the phonetic similarity as well, so that the overall similarity is computed as a convex
> combination of Wu and Palmer similarity and phonetic similarity (after being normalized) with coefficient
> of 0.5. Suggest an implementation and run your new code on selected MSH-WSD dataset.*


In [ ]:
import sys
sys.path.append("..")                                  
from wsd_utils import (ALL_ACRONYMS, ALL_TERMS, MAPPING, to_nltk, pywsd_form,
                       context_window, sample_instances, accuracy, mean)
import utils as u                                     

import json
import time
from pathlib import Path
import pandas as pd

import nltk
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)

from pywsd._wordnet import wn as pwn
from pywsd.similarity import sim
from pywsd.utils import lemmatize
from pywsd.tokenize import word_tokenize

Our new similarity measure combines the Wu-Palmer similarity and the phonetic similarity. Wu-Palmer is already between 0 and 1, but the Levenshtein distance between two Soundex codes goes from 0 to 4 (and a higher value means *more different*). So we first normalize it with `1 - distance / 4`: now it is also between 0 and 1, and 1 means the two words sound the same.

In [10]:
def phonetic_similarity(w1, w2):
    """Phonetic similarity between two words, from 0 (completely different) to 1 (same sound).
    Soundex codes always have 4 characters, so the Levenshtein distance goes from 0 to 4."""
    distance = u.levenshtein(u.soundex(w1), u.soundex(w2))
    return 1 - distance / 4

# Test
for w1, w2 in [("drug", "drugs"), ("two", "to"), ("drug", "truck"), ("drug", "clinic")]:
    print(f"{w1:5} {u.soundex(w1)}  {w2:7} {u.soundex(w2)}  similarity = {phonetic_similarity(w1, w2):.2f}")

drug  D620  drugs   D620  similarity = 1.00
two   T000  to      T000  similarity = 1.00
drug  D620  truck   T620  similarity = 0.75
drug  D620  clinic  C452  similarity = 0.00


Wu-Palmer compares two **senses** (synsets), while our phonetic similarity compares two **words**. To combine them, also the phonetic part must compare two senses. A sense can have more than one name (lemma), e.g. `['drug', 'dose']`. So the phonetic similarity between two senses is the **highest** phonetic similarity between a lemma of the first sense and a lemma of the second one: it is enough that one of the names sounds similar.

In [11]:
def sense_phonetic(s1, s2):
    """Phonetic similarity between two senses: the best value among all the pairs of their lemmas."""
    return max(phonetic_similarity(a, b) for a in s1.lemmas() for b in s2.lemmas())

clinic = pwn.synsets("clinic")[0]
medicine = pwn.synsets("medicine")[0]
for s in pwn.synsets("drug"):
    print(f"{str(s.lemmas()):22} vs clinic {sense_phonetic(s, clinic):.2f}   vs medicine {sense_phonetic(s, medicine):.2f}")

['drug']               vs clinic 0.00   vs medicine 0.25
['drug', 'dose']       vs clinic 0.00   vs medicine 0.25
['drug', 'do drugs']   vs clinic 0.25   vs medicine 0.25


We can notice that the phonetic similarity between senses can give points to the wrong sense only by chance (e.g. *do drugs* vs *clinic*). 

Now we use it inside a WSD method, to see how it changes the chosen sense.
The method works like the Wu-Palmer method of pywsd, but with our new measure `alpha * WuPalmer + (1 - alpha) * phonetic`:
1. for each sense of the target word, we compare it with all the senses of each context word and keep only the **best** value;
2. we **sum** these best values over all the context words;
3. the sense with the **highest sum** wins.

With `alpha = 1` it is only Wu-Palmer, with `alpha = 0.5` it is the measure asked by the task. The target word is removed from the context, otherwise it would compare with itself.


In [12]:
def wup_phonetic(context, word, alpha=0.5):
    ctx = [lemmatize(w) for w in word_tokenize(context)]
    target = lemmatize(word)
    ctx = [w for w in ctx if w.lower() != target.lower()]
    scores = []
    for i in pwn.synsets(target):
        total = 0.0
        for j in ctx:
            best = 0.0
            for k in pwn.synsets(j):
                wup = sim(i, k, "wup") or 0.0
                best = max(best, alpha * wup + (1 - alpha) * sense_phonetic(i, k))
            total += best
        scores.append((total, i))
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores


sentence = "I have been prescribed two important drugs today during my visit to clinic"
for alpha in (1, 0.5):
    print(f"alpha = {alpha}")
    for total, s in wup_phonetic(sentence, "drug", alpha):
        print(f"   {total:5.2f}  {str(s.lemmas()):22} {s.definition()}")

alpha = 1
    2.61  ['drug']               a substance that is used as a medicine or narcotic
    1.20  ['drug', 'do drugs']   use recreational drugs
    1.09  ['drug', 'dose']       administer a drug to
alpha = 0.5
    2.95  ['drug', 'dose']       administer a drug to
    2.71  ['drug']               a substance that is used as a medicine or narcotic
    2.50  ['drug', 'do drugs']   use recreational drugs


Wu-Palmer alone is correct, but with the phonetic part the verb sense *administer a drug to* wins. This sense has two lemmas, *drug* and *dose*, and *dose* has a short Soundex code (D200) similar to the codes of short context words (*two*, *to*, *today* → T000, T300). So it gets a small bonus from many context words, only because of the sound.

Now we evaluate the new similarity on the same 39 words of Task 4, with the same sample of abstracts (10 per sense, same seed) and the same context window (10 words on each side). To compare, we try different weights for Wu-Palmer: `alpha = 1` (only Wu-Palmer), `alpha = 0.8` and `alpha = 0.5` (the measure asked by the task).

In [13]:
ACRONYMS = [w for w in ALL_ACRONYMS if w in MAPPING]
TERMS    = [w for w in ALL_TERMS if w in MAPPING]

print(len(ACRONYMS), "acronyms,", len(TERMS), "terms")
print("Abstracts in the sample:", sum(len(sample_instances(w)) for w in ACRONYMS + TERMS))

20 acronyms, 19 terms
Abstracts in the sample: 810


`wsd_alphas` does the same as `wup_phonetic`, but for all the alphas at once: Wu-Palmer and the phonetic similarity do not depend on alpha, so we compute them only once and then combine them with each alpha. This makes the evaluation about three times faster. It returns only the winning sense for each alpha, translated into its NLTK name to compare it with `MAPPING`. Words with no sense in WordNet are skipped, since every answer would be wrong anyway.

In [14]:
ALPHAS = [1, 0.8, 0.5]


def wsd_alphas(context, word):
    """Best sense for each alpha. Wu-Palmer and phonetic values are computed only once and reused."""
    ctx = [lemmatize(w) for w in word_tokenize(context)]
    target = lemmatize(word)
    ctx = [w for w in ctx if w.lower() != target.lower()]
    totals = {a: [] for a in ALPHAS}
    for i in pwn.synsets(target):
        pairs = [[(sim(i, k, "wup") or 0.0, sense_phonetic(i, k)) for k in pwn.synsets(j)] for j in ctx]
        for a in ALPHAS:
            total = sum(max([a * w + (1 - a) * p for w, p in pairs_j], default=0.0) for pairs_j in pairs)
            totals[a].append((total, i))
    return {str(a): to_nltk(max(t, key=lambda x: x[0])[1]) if t else None for a, t in totals.items()}

In [15]:
CACHE6 = Path("task6_predictions.json")
predictions = json.loads(CACHE6.read_text()) if CACHE6.exists() else {}

for word in ACRONYMS + TERMS:
    if word in predictions:
        continue
    t = time.time()
    covered = any(MAPPING[word].values())
    rows = []
    for inst in sample_instances(word):
        if covered:
            answers = wsd_alphas(context_window(inst["tagged"]), pywsd_form(word))
        else:
            answers = {str(a): None for a in ALPHAS}
        rows.append({"cui": inst["cui"], "answers": answers})
    predictions[word] = rows
    CACHE6.write_text(json.dumps(predictions))
    print(f"{word:16} {len(rows):3} abstracts {time.time() - t:6.1f} s")

print("Done.")

Done.


In [16]:
pred4 = json.loads(Path("../p3_4_pywsd/task4_predictions.json").read_text())

compare = [("Wu-Palmer (pywsd, Task 4)", pred4, "Wu-Palmer")]
compare += [(f"alpha = {a}", predictions, str(a)) for a in ALPHAS]

table = []
for name, preds, method in compare:
    line = {"method": name}
    for group_name, group in [("acronyms", ACRONYMS), ("terms", TERMS)]:
        line[f"{group_name} acc"] = mean(accuracy(preds[w], w, method) for w in group)
        line[f"{group_name} cov"] = mean(accuracy(preds[w], w, method, covered_only=True) for w in group)
    table.append(line)

results6 = pd.DataFrame(table).set_index("method")
results6.round(3)

,acronyms acc,acronyms cov,terms acc,terms cov
method,,,,
"Wu-Palmer (pywsd, Task 4)",0.250,0.905,0.345,0.562
alpha = 1,0.248,0.895,0.345,0.562
alpha = 0.8,0.249,0.895,0.332,0.547
alpha = 0.5,0.265,0.920,0.342,0.559


**Observations:**
- `alpha = 0.5` is a bit better on acronyms (0.248 → 0.265), but on terms the best is still `alpha = 1` (0.345).
- The differences are very small and `alpha = 0.8` is not between the other two, so they depend on a few abstracts and are not significant.
- So, on average, adding the phonetic similarity does not really improve Wu-Palmer.

The averages hide what happens on each word. Here we print only the words where the alpha changes the accuracy, to see where the phonetic similarity helps and where it makes things worse.

In [17]:
for w in ACRONYMS + TERMS:
    accs = {a: accuracy(predictions[w], w, str(a)) for a in ALPHAS}
    if len(set(accs.values())) > 1:          # only the words where alpha changes the result
        print(f"{w:12} " + "  ".join(f"alpha={a}: {accs[a]:.2f}" for a in ALPHAS))

ADP          alpha=1: 0.70  alpha=0.8: 0.65  alpha=0.5: 0.85
Ca           alpha=1: 0.33  alpha=0.8: 0.30  alpha=0.5: 0.30
PVC          alpha=1: 0.60  alpha=0.8: 0.70  alpha=0.5: 0.85
RA           alpha=1: 0.33  alpha=0.8: 0.33  alpha=0.5: 0.30
Cell         alpha=1: 0.20  alpha=0.8: 0.10  alpha=0.5: 0.25
Cement       alpha=1: 0.40  alpha=0.8: 0.45  alpha=0.5: 0.45
Cilia        alpha=1: 0.20  alpha=0.8: 0.45  alpha=0.5: 0.50
ERUPTION     alpha=1: 0.25  alpha=0.8: 0.00  alpha=0.5: 0.00
Exercises    alpha=1: 0.00  alpha=0.8: 0.05  alpha=0.5: 0.10
Milk         alpha=1: 0.60  alpha=0.8: 0.60  alpha=0.5: 0.55
Nurse        alpha=1: 0.20  alpha=0.8: 0.05  alpha=0.5: 0.15
Respiration  alpha=1: 0.80  alpha=0.8: 0.70  alpha=0.5: 0.65
Ventricles   alpha=1: 0.50  alpha=0.8: 0.50  alpha=0.5: 0.45


**Observations:**
- The phonetic part changes the result only on 13 of the 39 words, and it helps on some words and makes things worse on others.
- It helps a lot on two acronyms, **ADP** (0.70 → 0.85) and **PVC** (0.60 → 0.85), and this is the reason why acronyms improve on average. A Soundex code has only 4 characters, so the code of a long name is the same as the code of its first word: *adenosine diphosphate* = *adenosine* (A352), *premature ventricular contraction* = *premature* (P653). When the abstract contains *adenosine* or *premature*, the correct sense gets a phonetic similarity of 1.
- It makes things worse on terms like **ERUPTION** (0.25 → 0.00) and **Respiration** (0.80 → 0.65): here all the senses have the same name, so the phonetic part cannot separate them and only adds noise, as with *dose* in the *drug* example.
- **Conclusion:** the phonetic similarity is not related to meaning. It helps only when the name of the correct sense contains a word of the context, so it is not a good general improvement for WSD.